In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
     

In [2]:
df = pd.read_csv("qoute_dataset.csv")

In [3]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
df.shape

(3038, 2)

In [5]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

In [6]:
# Convert in lower case
quotes = quotes.str.lower()

In [7]:
# Remove Punchuations 
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))
     

In [8]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: str

### Tokenization

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [10]:
vocab_size = 8978

tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)
     


In [11]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [12]:
sequence = tokinizer.texts_to_sequences(quotes)

In [13]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [14]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [15]:
for i in range(3):
  print(quotes[i])
     

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [16]:

for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


### Creating Input Output Variable

In [17]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [18]:
len(X)

85270

In [19]:
len(y)

85270

### Padding Apply for same size

In [20]:
max_len = max(len(x) for x in X)
print(max_len)
     

745


In [21]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')
     

In [22]:
y = np.array(y)

In [23]:
X_padded.shape

(85270, 745)

### ONE Hot Encoding on y 

In [24]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [25]:
y.shape

(85270,)

In [26]:
y_one_hot.shape     

(85270, 8978)

### Creating Models

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [28]:
embedding_dim = 50
rnn_units = 128
     

### RNN Model

In [29]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))


C:\Users\KHUSI SINGH\miniconda3\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [30]:
# compile rnn_model
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
     

In [31]:
rnn_model.summary()
     

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
epochs = 10 
batch_size = 128

In [33]:
history_rnn = rnn_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 390s 641ms/step - accuracy: 0.0443 - loss: 6.7170 - val_accuracy: 0.0572 - val_loss: 6.5540
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 379s 632ms/step - accuracy: 0.0751 - loss: 6.1420 - val_accuracy: 0.0896 - val_loss: 6.3160
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 385s 642ms/step - accuracy: 0.0995 - loss: 5.7951 - val_accuracy: 0.1005 - val_loss: 6.2539
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 832s 795ms/step - accuracy: 0.1172 - loss: 5.4962 - val_accuracy: 0.1080 - val_loss: 6.2832
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 389s 649ms/step - accuracy: 0.1316 - loss: 5.2402 - val_accuracy: 0.1112 - val_loss: 6.3200
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 393s 655ms/step - accuracy: 0.1428 - loss: 5.0071 - val_accuracy: 0.1116 - val_loss: 6.3670
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 396s 659ms/step - accuracy: 0.1557 - loss: 4.7913 - val_accuracy: 0.1105 - val_loss: 6.4481
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 442s 659ms/step - accuracy: 0.1699 -

### LSTM Model

In [34]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))
     

In [35]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [36]:
lstm_model.summary()
     

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [41]:
epochs = 10 
batch_size = 128

In [44]:
history_lstm = lstm_model.fit(
    X_padded,
    y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
    
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 897s 1s/step - accuracy: 0.0396 - loss: 6.7497 - val_accuracy: 0.0504 - val_loss: 6.6726
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 893s 1s/step - accuracy: 0.0599 - loss: 6.3039 - val_accuracy: 0.0672 - val_loss: 6.5236
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 922s 1s/step - accuracy: 0.0845 - loss: 6.0268 - val_accuracy: 0.0878 - val_loss: 6.4468
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 902s 2s/step - accuracy: 0.0996 - loss: 5.8179 - val_accuracy: 0.0926 - val_loss: 6.4209
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 979s 2s/step - accuracy: 0.1094 - loss: 5.6414 - val_accuracy: 0.1014 - val_loss: 6.4084
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 990s 2s/step - accuracy: 0.1194 - loss: 5.4750 - val_accuracy: 0.1065 - val_loss: 6.4182
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 961s 2s/step - accuracy: 0.1290 - loss: 5.3239 - val_accuracy: 0.1073 - val_loss: 6.4403
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 1075s 2s/step - accuracy: 0.1346 - loss: 5.1839 - val_acc

In [57]:

lstm_model.save("lstm_model.h5")

In [58]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.h5")

In [59]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word
     

In [60]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
     

In [61]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [63]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokinizer,seed_text,max_len)
print(next_word)
     
#worrying

are


In [64]:
def generate_text(model,tokenizer,seed_Stext,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [66]:
seed = "are you a "
generate_text = generate_text(lstm_model,tokinizer,seed,max_len,10)
print(generate_text)
     
# are you a  thousand times i wrote the less if it does not

are you a  man who will be a man who will be a


In [67]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokinizer, f)

In [68]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)
     